# Build your own poet — a small language model from one text file

This notebook is the *build-your-own* companion to **How a language model works** (`how-llms-work.html`). It builds the exact same kind of machine that writes Thirukkural on that page, from scratch, on **any text you give it** — Bharathiyar, Kabir, Shakespeare, film lyrics, your own writing.

**You do not need to know how to program.** Run the cells top to bottom (`Runtime → Run all`). The only decision is which text file to use.

| Step | What happens | Chapter on the page |
|---|---|---|
| 1 | Get a book: upload a `.txt`, or use the built-in Thirukkural / Shakespeare sample | 2 · the only book |
| 2 | Every letter gets a number | 3 · letters → numbers |
| 3 | The education: guess the next letter, get corrected, nudge — a few thousand times | 4–8 |
| 4 | Teach a habit: `# label → example` | 9 · teaching a habit |
| 5 | A judge: pairs of attempts, keep what the judge prefers | 10 · a judge, not a teacher |
| 6 | Write new poems, and save the weights in the page's format | 11–12 |

Takes ~10 minutes on the free Colab CPU, ~2 minutes with a GPU (`Runtime → Change runtime type → T4 GPU`).


In [ ]:
#@title Step 1 · The book  { display-mode: "form" }
#@markdown Choose a sample, or set `use_upload` to True and pick a `.txt` file from your computer.
#@markdown The file should have one example (poem, verse, recipe…) per paragraph, with a blank line between examples.
sample = "thirukkural"  #@param ["thirukkural", "shakespeare-sonnets"]
use_upload = False  #@param {type:"boolean"}

import re, json, urllib.request, unicodedata
if use_upload:
    from google.colab import files
    up = files.upload()
    name = list(up.keys())[0]
    text = up[name].decode('utf-8')
elif sample == "thirukkural":
    raw = json.load(urllib.request.urlopen('https://raw.githubusercontent.com/tk120404/thirukkural/master/thirukkural.json'))['kural']
    fix = lambda s: s.replace('஦வ்ருஉம்', 'வெருஉம்').replace('அள஧ க்கும்', 'அளிக்கும்').replace('஧', 'ி')
    text = ''.join(fix(k['Line1']).strip() + '\n' + fix(k['Line2']).strip() + '\n\n' for k in raw)
    # labels for step 4: the chapter of each couplet
    det = json.load(urllib.request.urlopen('https://raw.githubusercontent.com/tk120404/thirukkural/master/detail.json'))
    LABELS = {}
    for paal in det[0]['section']['detail']:
        for iyal in paal['chapterGroup']['detail']:
            for ch in iyal['chapters']['detail']:
                for i in range(ch['start'], ch['end'] + 1): LABELS[i - 1] = ch['name']
else:
    raw = urllib.request.urlopen('https://www.gutenberg.org/cache/epub/1041/pg1041.txt').read().decode('utf-8').replace('\r', '')
    body = raw.split("*** START OF THE PROJECT GUTENBERG EBOOK SHAKESPEARE'S SONNETS ***")[1].split('*** END OF')[0]
    body = body.replace('’', "'").replace('‘', "'").replace('“', '"').replace('”', '"')
    son, cur = [], []
    for ln in body.split('\n'):
        s = ln.strip()
        if re.fullmatch(r'[IVXLC]+', s):
            if cur: son.append(cur); cur = []
        elif s and not s.startswith('THE SONNETS') and not s.startswith('by William'): cur.append(s)
    if cur: son.append(cur)
    text = ''.join('\n'.join(s) + '\n\n' for s in son)

text = unicodedata.normalize('NFC', text)
examples = [e.strip() for e in text.split('\n\n') if e.strip()]
if 'LABELS' not in dir(): LABELS = {i: e.split()[0] for i, e in enumerate(examples)}   # default label: the first word
print(f'{len(examples):,} examples · {len(text):,} letters · {len(set(text))} different letters')
print('first example:\n' + examples[0])


In [ ]:
#@title Step 2 · The machine (the same 120 lines that built the page's models)  { display-mode: "form" }
#@markdown Run this cell once. It defines the model, the training loop, and the export to the page's format.
"""Minimal character-level GPT (nanoGPT-style). Also used verbatim as the 'build your own' recipe on the page."""
import math, json, time, random
import torch, torch.nn as nn, torch.nn.functional as F

import os; torch.set_num_threads(os.cpu_count() or 2)

class Config:
    def __init__(self, vocab_size, block_size=128, n_layer=3, n_head=4, n_embd=64, dropout=0.15):
        self.vocab_size, self.block_size, self.n_layer, self.n_head, self.n_embd, self.dropout = \
            vocab_size, block_size, n_layer, n_head, n_embd, dropout

class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head, self.n_embd = c.n_head, c.n_embd
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.proj = nn.Linear(c.n_embd, c.n_embd)
        self.drop = nn.Dropout(c.dropout)
        self.register_buffer('mask', torch.tril(torch.ones(c.block_size, c.block_size)).view(1, 1, c.block_size, c.block_size))
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = (self.drop(att) @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(y))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1 = nn.LayerNorm(c.n_embd)
        self.attn = Attention(c)
        self.ln2 = nn.LayerNorm(c.n_embd)
        self.mlp = nn.Sequential(nn.Linear(c.n_embd, 4 * c.n_embd), nn.GELU(), nn.Linear(4 * c.n_embd, c.n_embd), nn.Dropout(c.dropout))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos_emb = nn.Embedding(c.block_size, c.n_embd)
        self.drop = nn.Dropout(c.dropout)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T)))
        for b in self.blocks: x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new, temperature=1.0, top_k=None, stop=None):
        self.eval()
        for _ in range(max_new):
            logits, _ = self(idx[:, -self.c.block_size:])
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
            idx = torch.cat([idx, nxt], dim=1)
            if stop is not None and stop(idx[0].tolist()): break
        return idx

def n_params(m): return sum(p.numel() for p in m.parameters())

# ---------------- data ----------------
class Data:
    def __init__(self, text, vocab, val_frac=0.1, seed=1, unit_sep='\n\n'):
        self.vocab = vocab
        self.stoi = {ch: i for i, ch in enumerate(vocab)}
        self.itos = {i: ch for ch, i in self.stoi.items()}
        units = [u for u in text.split(unit_sep) if u.strip()]
        rng = random.Random(seed); rng.shuffle(units)
        nv = max(1, int(len(units) * val_frac))
        self.val_units, self.train_units = units[:nv], units[nv:]
        self.train = torch.tensor(self.encode(unit_sep.join(self.train_units) + unit_sep), dtype=torch.long)
        self.val = torch.tensor(self.encode(unit_sep.join(self.val_units) + unit_sep), dtype=torch.long)
    def encode(self, s): return [self.stoi[c] for c in s]
    def decode(self, l): return ''.join(self.itos[i] for i in l)
    def batch(self, split, B, T):
        d = self.train if split == 'train' else self.val
        ix = torch.randint(len(d) - T, (B,))
        return torch.stack([d[i:i + T] for i in ix]), torch.stack([d[i + 1:i + 1 + T] for i in ix])

@torch.no_grad()
def eval_loss(model, data, B=32, iters=20):
    model.eval(); out = {}
    for split in ('train', 'val'):
        ls = []
        for _ in range(iters):
            x, y = data.batch(split, B, model.c.block_size)
            _, l = model(x, y); ls.append(l.item())
        out[split] = sum(ls) / len(ls)
    model.train(); return out

def sample_text(model, data, prompt, n=200, temperature=0.8, seed=0, top_k=None, stop=None):
    torch.manual_seed(seed)
    idx = torch.tensor([data.encode(prompt)], dtype=torch.long)
    out = model.generate(idx, n, temperature=temperature, top_k=top_k, stop=stop)
    return data.decode(out[0].tolist())

def train(model, data, steps, lr=1e-3, B=32, log_every=50, milestones=(), sample_prompt='\n', sample_seed=0, sample_n=200, min_lr_frac=0.1, tag=''):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    log, samples = [], {}
    t0 = time.time()
    for step in range(steps + 1):
        if step in milestones:
            samples[step] = sample_text(model, data, sample_prompt, n=sample_n, seed=sample_seed)
            model.train()
        if step % log_every == 0 or step == steps:
            e = eval_loss(model, data)
            log.append(dict(step=step, train=round(e['train'], 4), val=round(e['val'], 4)))
            print(f'{tag} step {step:5d} train {e["train"]:.3f} val {e["val"]:.3f}  {time.time()-t0:.0f}s', flush=True)
        if step == steps: break
        cur_lr = min_lr_frac * lr + 0.5 * (1 - min_lr_frac) * lr * (1 + math.cos(math.pi * step / steps))
        for g in opt.param_groups: g['lr'] = cur_lr
        x, y = data.batch('train', B, model.c.block_size)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    return log, samples

# ---------------- export (int8 per-tensor for the browser) ----------------
import base64, numpy as np
def export(model, vocab, path, extra=None):
    c = model.c
    tensors = {}
    for name, p in model.state_dict().items():
        if name.endswith('.mask'): continue
        w = p.detach().float().numpy()
        if w.ndim == 2 and w.size > 4096:  # big matrices -> int8
            scale = float(np.abs(w).max() / 127.0) or 1.0
            q = np.clip(np.round(w / scale), -127, 127).astype(np.int8)
            tensors[name] = dict(shape=list(w.shape), dtype='i8', scale=scale, data=base64.b64encode(q.tobytes()).decode())
        else:  # small vectors -> float16
            tensors[name] = dict(shape=list(w.shape), dtype='f16', data=base64.b64encode(w.astype(np.float16).tobytes()).decode())
    out = dict(config=dict(vocab_size=c.vocab_size, block_size=c.block_size, n_layer=c.n_layer, n_head=c.n_head, n_embd=c.n_embd),
               vocab=vocab, n_params=n_params(model), tensors=tensors)
    if extra: out.update(extra)
    json.dump(out, open(path, 'w'), ensure_ascii=False, separators=(',', ':'))
    return out

def load_exported(path):
    """Rebuild a torch model from the exported json (to verify the quantisation round-trip)."""
    j = json.load(open(path))
    cf = j['config']; c = Config(cf['vocab_size'], cf['block_size'], cf['n_layer'], cf['n_head'], cf['n_embd'], 0.0)
    m = GPT(c); sd = m.state_dict()
    for name, t in j['tensors'].items():
        raw = base64.b64decode(t['data'])
        if t['dtype'] == 'i8': w = np.frombuffer(raw, dtype=np.int8).astype(np.float32) * t['scale']
        else: w = np.frombuffer(raw, dtype=np.float16).astype(np.float32)
        sd[name] = torch.tensor(w.reshape(t['shape']))
    m.load_state_dict(sd); m.eval(); return m, j['vocab']


vocab = sorted(set(text) | {'#'})
print('vocabulary (every letter gets a locker number):')
print({i: (c if c not in ('\n', ' ') else repr(c)) for i, c in enumerate(vocab)})


In [ ]:
#@title Step 3 · The education  { display-mode: "form" }
#@markdown Watch the *surprise* (loss) fall, and read what the machine writes at each milestone. Stop early if `val` starts rising.
steps = 3000  #@param {type:"slider", min:500, max:6000, step:500}
numbers_per_letter = 64  #@param [64, 96, 128] {type:"raw"}
rounds = 3  #@param [2, 3, 4, 6] {type:"raw"}

data = Data(text, vocab)
torch.manual_seed(42)
model = GPT(Config(len(vocab), block_size=128, n_layer=rounds, n_head=4, n_embd=numbers_per_letter, dropout=0.15))
print(f'{n_params(model):,} dials · {len(data.train):,} letters to learn from · {len(data.val):,} held out')
log, samples = train(model, data, steps, lr=1e-3, milestones=(0, 100, 500, 1000, 2000, steps), sample_prompt='\n', sample_n=250, tag='education')
for s, txt in samples.items():
    print(f'\n===== what it wrote at step {s} =====\n{txt.strip()[:400]}')
export(model, vocab, 'base.json', extra=dict(name='base', loss=log, samples={str(k): v for k, v in samples.items()}))
print('\nsaved base.json')


In [ ]:
#@title Step 4 · Teach a habit (fine-tuning): `# label` → example  { display-mode: "form" }
#@markdown Each example is prefixed with a label line. For Thirukkural the label is the chapter; for your own file it is the first word of each example (change `LABELS` above to use anything you like — a mood, an author, a topic).
sft_text = ''.join('# ' + LABELS[i] + '\n' + e + '\n\n' for i, e in enumerate(examples))
sft_data = Data(sft_text, vocab)
log2, samples2 = train(model, sft_data, 1500, lr=5e-4, milestones=(0, 1500), sample_prompt='# ' + LABELS[0] + '\n', sample_n=150, tag='habit')
print('\nbefore fine-tuning, given the label:\n' + samples2[0][:200]); print('\nafter:\n' + samples2[1500][:200])
export(model, vocab, 'sft.json', extra=dict(name='sft', loss=log2))
print('\nsaved sft.json')


In [ ]:
#@title Step 5 · A judge (preference optimisation)  { display-mode: "form" }
#@markdown Write your own judge below: a function that returns True for a good example. The default checks the Thirukkural shape (4 words, then 3, ending with a full stop) — change it for your text (e.g. `len(lines) == 14` for sonnets, or "contains the label's word").
import copy, random, torch.nn.functional as F

def judge(out):
    body = out.split('\n\n')[0]; lines = body.split('\n')
    return len(lines) == 2 and len(lines[0].split()) == 4 and len(lines[1].split()) == 3 and body.endswith('.')

def stop(ids): return len(ids) > 1 and ids[-1] == ids[-2] == sft_data.stoi['\n']
def logp(m, prompt, resp):
    ids = sft_data.encode(prompt + resp); x = torch.tensor([ids[:-1]]); y = torch.tensor([ids[1:]])
    lp = F.log_softmax(m(x)[0][0], -1).gather(1, y[0][:, None]).squeeze(1)
    return lp[len(sft_data.encode(prompt)) - 1:].sum()
def pass_rate(m, n=150):
    return sum(judge(sample_text(m, sft_data, '# ' + LABELS[random.randrange(len(examples))] + '\n', n=120, temperature=1.0, seed=i, stop=stop).split('\n', 1)[1]) for i in range(n)) / n

ref = copy.deepcopy(model).eval()
print('before the judge: %.0f%% of attempts pass' % (100 * pass_rate(ref)))
pairs = []
for i in range(400):
    p = '# ' + LABELS[random.randrange(len(examples))] + '\n'
    a, b = [sample_text(model, sft_data, p, n=120, temperature=1.0, seed=7000 + 2 * i + k, stop=stop)[len(p):] for k in (0, 1)]
    if judge(a) != judge(b): pairs.append((p,) + ((a, b) if judge(a) else (b, a)))
print(f'{len(pairs)} pairs the judge could separate')
assert len(pairs) >= 8, 'The judge could not separate enough pairs — train longer in step 3, or loosen the judge.'
with torch.no_grad(): refs = [(logp(ref, p, w).item(), logp(ref, p, l).item()) for p, w, l in pairs]
opt = torch.optim.AdamW(model.parameters(), lr=2e-5); beta = 0.5
model.train()
for step in range(60):
    loss = 0
    for i in random.sample(range(len(pairs)), min(8, len(pairs))):
        p, w, l = pairs[i]; rw, rl = refs[i]
        loss = loss - F.logsigmoid(beta * ((logp(model, p, w) - rw) - (logp(model, p, l) - rl)))
    opt.zero_grad(); (loss / 8).backward(); opt.step()
model.eval()
print('after the judge:  %.0f%% of attempts pass' % (100 * pass_rate(model)))
export(model, vocab, 'judged.json', extra=dict(name='judged'))
print('saved judged.json')


In [ ]:
#@title Step 6 · Write poems, and download the weights  { display-mode: "form" }
label = LABELS[0]  #@param {type:"string"}
temperature = 0.8  #@param {type:"slider", min:0.3, max:1.5, step:0.1}
how_many = 5  #@param {type:"slider", min:1, max:20, step:1}
for i in range(how_many):
    out = sample_text(model, sft_data, '# ' + label + '\n', n=150, temperature=temperature, seed=1000 + i, stop=stop)
    print(out.strip() + '\n' + '-' * 40)

from google.colab import files
for f in ('base.json', 'sft.json', 'judged.json'): files.download(f)


## Put it on a page

The three `.json` files are in the exact format `how-llms-work.html` reads. Open that page, scroll to chapter 12, **Load the poet you built**, and choose any of the three files — it runs in your browser immediately, on the same engine as the Thirukkural machine; nothing is uploaded. `base.json` continues any text; `sft.json` answers a `# label` line; `judged.json` is the version the judge shaped.

To publish a page of your own, copy `how-llms-work.html`, find `const DATA = …` in its source and swap the `models` entries for your files (the JavaScript engine right above it, `Engine.Model`, runs any machine of this shape and needs no changes).

## What to try next

* **More text.** This is the single biggest lever. A 1 MB book (roughly a novel) supports `numbers_per_letter = 128` and `rounds = 6`; expect real sentences.
* **A better judge.** The default judges shape only. Judge meaning too (does the poem mention the label? is it novel? does a word list say it is positive?) and the machine will chase that instead — carefully; it will chase whatever you reward.
* **Two languages.** Put the English meaning *and* the Tamil verse in each example, with the English as the label line, and the machine learns to write Tamil from an English request. It needs a bigger machine and more steps.
